In [1]:
%%capture
!pip install transformers datasets accelerate bitsandbytes sentence-transformers spacy rouge_score bert_score langchain langchain-community langchain-huggingface huggingface_hub "numpy<2.0" "scipy>=1.10"
!python -m spacy download it_core_news_sm

In [ ]:
import os
import json
import gc
import shutil
import torch
import torch.nn as nn
import spacy
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    Trainer, 
    TrainingArguments
)
from huggingface_hub import login, HfApi

# On Kaggle
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
HF_TOKEN = UserSecretsClient().get_secret(secret_label)

#HF_TOKEN = os.getenv("HF_TOKEN")
HF_USERNAME = "LookUpMark"
REPO_NAME = "sigext-wits-it-15k"

CONFIG = {
    "SBERT_MODEL": 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    "LONGFORMER_MODEL": "markussagen/xlm-roberta-longformer-base-4096",
    "MAX_LEN": 2048,
    "SEMANTIC_THRESHOLD": 0.60,
    "NUM_TRAIN_SAMPLES": 15000,
    "TRAIN_FILE": "wits_train_15k.jsonl",
    "OUTPUT_DIR": "./sigext_15k_final",
    "MIN_SOURCE_LEN": 500,
    "MAX_SOURCE_LEN": 10000,
    "MIN_SUMMARY_LEN": 50,
    "MIN_SENT_LEN": 20,
    "BATCH_SIZE": 32
}

In [3]:
# Data Engineering
def prepare_data_15k():
    # Setup NLP
    try:
        nlp = spacy.load("it_core_news_sm")
    except:
        nlp = spacy.load("it_core_news_sm")

    # Load SBERT model
    print("="*60 + "\nLoading SBERT model...\n" + "="*60)
    print("\n\n\n")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sbert = SentenceTransformer(CONFIG["SBERT_MODEL"], device=device)
    
    # Load dataset
    print("="*60 + "\nLoading dataset...\n" + "="*60)
    print("\n\n\n")
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    
    # Labeling
    print("="*60 + "\nLabeling...\n" + "="*60)
    print("\n\n\n")
    count = 0
    with open(CONFIG["TRAIN_FILE"], "w") as f_out:
        pbar = tqdm(total=CONFIG["NUM_TRAIN_SAMPLES"], desc="Labeling")
        
        # Iterate over dataset
        for entry in dataset:
            source = entry['source']
            summary = entry['summary']
            
            # Filter when source or summary is too short or too long
            if len(source) < CONFIG["MIN_SOURCE_LEN"] or len(summary) < CONFIG["MIN_SUMMARY_LEN"] or len(source) > CONFIG["MAX_SOURCE_LEN"]:
                continue

            # Tokenize sentences
            doc_sents = [s.text for s in nlp(source).sents if len(s.text) > CONFIG["MIN_SENT_LEN"]]
            sum_sents = [s.text for s in nlp(summary).sents if len(s.text) > CONFIG["MIN_SENT_LEN"]]

            # Filter when no sentences are found
            if not doc_sents or not sum_sents:
                continue

            # Compute embeddings in batches
            doc_emb = sbert.encode(doc_sents, convert_to_tensor=True, show_progress_bar=False, batch_size=CONFIG["BATCH_SIZE"])
            sum_emb = sbert.encode(sum_sents, convert_to_tensor=True, show_progress_bar=False, batch_size=CONFIG["BATCH_SIZE"])
            scores = util.cos_sim(doc_emb, sum_emb)
            
            # Labeling sentences
            labels = []
            for i in range(len(doc_sents)):
                # Compute max score
                max_score = scores[i].max().item()
                # Label
                labels.append(1 if max_score > CONFIG["SEMANTIC_THRESHOLD"] else 0)
            
            # Write to file
            if 1 in labels:
                f_out.write(json.dumps({"sentences": doc_sents, "labels": labels}) + "\n")
                count += 1
                pbar.update(1)
            
            # Stop when enough samples are collected
            if count >= CONFIG["NUM_TRAIN_SAMPLES"]:
                break
        
        pbar.close()

    print("="*60 + "\nLabeling completed!\n" + "="*60)
    
    # Cleanup
    del sbert; torch.cuda.empty_cache(); gc.collect()

In [4]:
class SigExtDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = [json.loads(line) for line in open(path)]
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    # Returns a dictionary with input_ids, attention_mask and labels
    def __getitem__(self, idx):
        # Join sentences
        item = self.data[idx]
        text = " ".join(item['sentences'])
        labels = item['labels']

        # Tokenize
        enc = self.tokenizer(text, truncation=True, max_length=CONFIG["MAX_LEN"], padding="max_length")

        # We set -100 as padding label in order to ignore it when computing the loss
        token_labels = [-100] * len(enc['input_ids'])

        # Limit labels to max length
        limit = min(len(labels), CONFIG["MAX_LEN"])
        token_labels[:limit] = labels[:limit]

        return {"input_ids": torch.tensor(enc['input_ids']), "attention_mask": torch.tensor(enc['attention_mask']), "labels": torch.tensor(token_labels)}

class WeightedTrainer(Trainer):
    # Compute loss
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Get labels and compute loss
        labels = inputs.get("labels")

        # Compute outputs
        outputs = model(**inputs)
        device = inputs["input_ids"].device

        # Compute weighted loss
        class_weights = torch.tensor([1.0, 10.0]).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        loss = criterion(
            outputs.get("logits").view(-1, 2),
            labels.view(-1)
        )
        
        return (loss, outputs) if return_outputs else loss

In [5]:
def train_sigext_15k():
    # Load Longformer model
    print("="*60 + "\nLoading Longformer model...\n" + "="*60)
    print("\n\n\n")
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["LONGFORMER_MODEL"])
    model = AutoModelForTokenClassification.from_pretrained(CONFIG["LONGFORMER_MODEL"], num_labels=2)

    # Training arguments
    args = TrainingArguments(
        output_dir="./checkpoints",
        num_train_epochs=2,
        per_device_train_batch_size=2,   
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        fp16=True,
        save_strategy="epoch",
        logging_steps=100,
        report_to="none"
    )
    
    # Dataset
    dataset = SigExtDataset(
        CONFIG["TRAIN_FILE"],
        tokenizer
    )
    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=dataset
    )
    
    # Training
    print("="*60 + "\nTraining...\n" + "="*60)
    print("\n\n\n")
    trainer.train()
    
    # Save model
    print("="*60 + "\nSaving model...\n" + "="*60)
    print("\n\n\n")
    model.save_pretrained(CONFIG["OUTPUT_DIR"])
    tokenizer.save_pretrained(CONFIG["OUTPUT_DIR"])

    print("="*60 + "\nTraining completed!\n" + "="*60)
    
    # Cleanup
    del model, trainer; torch.cuda.empty_cache(); gc.collect()

In [6]:
def upload_to_hub():
    # Uploading to Hugging Face
    repo_id = f"{HF_USERNAME}/{REPO_NAME}"
    
    api = HfApi()
    
    try:
        # Try to create or overwrite a repository
        api.create_repo(repo_id=repo_id, exist_ok=True)
        print("="*60 + "\nRepository successfully created!\n" + "="*60)
        print("\n\n\n")
    except Exception as e:
        print(f"   ERROR while creating the repository: {e}")
        return

    # Uploading folder
    try:
        api.upload_folder(
            folder_path=CONFIG["OUTPUT_DIR"],
            repo_id=repo_id,
            repo_type="model",
            commit_message="Training completed with 15k samples"
        )
        print("="*60 + "\nFolder successfully uploaded to Hugging Face!\n" + "="*60)
        print("\n\n\n")
    except Exception as e:
        print(f"   ERROR while uploading the folder: {e}")

In [7]:
# Login
login(token=HF_TOKEN)

In [8]:
# Prepare data
prepare_data_15k()

Loading SBERT model...






modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Labeling...






Labeling:   0%|          | 0/15000 [00:00<?, ?it/s]

Labeling completed!


In [9]:
# Train model
train_sigext_15k()

Loading Longformer model...






tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/773 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Training...






/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.598300
200,0.600700
300,0.587000
400,0.586200
500,0.573700
600,0.575300
700,0.572500
800,0.569100
900,0.572400
1000,0.559000


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Saving model...




Training completed!


In [10]:
# Upload to Hugging Face Hub
upload_to_hub()

Repository successfully created!






Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Folder successfully uploaded to Hugging Face!




